# Milestone 4

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import Dataset

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
LABELS = ['A','B','C','D','E']
label2idx = {l:i for i,l in enumerate(LABELS)}

## Q1: Label Encoding

In [ ]:
train['label'] = train['answer'].map(label2idx)
print('Encoded label at index 150:', train.iloc[150]['label'])  # 2

## Q2: Prompt-Option Formatting

In [ ]:
row0 = train.iloc[0]
formatted = str(row0['prompt']) + ' [SEP] ' + str(row0['B'])
print('Character length:', len(formatted))  # 407

## Q3-Q4: MCQ Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_mcq_row(row, max_length=128):
    prompt = str(row['prompt'])
    ids_list, mask_list = [], []
    for label in LABELS:
        enc = tokenizer(prompt, str(row[label]),
                       max_length=max_length, padding='max_length',
                       truncation=True, return_tensors='pt')
        ids_list.append(enc['input_ids'].squeeze(0))
        mask_list.append(enc['attention_mask'].squeeze(0))
    return torch.stack(ids_list), torch.stack(mask_list)

ids, mask = tokenize_mcq_row(train.iloc[0])
print('Single row shape:', ids.unsqueeze(0).shape)  # [1, 5, 128]
print('Second dimension:', ids.shape[0])  # 5

# Batch of 16
batch_ids = torch.stack([tokenize_mcq_row(train.iloc[i])[0] for i in range(16)])
print('Batch shape:', batch_ids.shape)  # [16, 5, 128]
print('Total token positions:', 16 * 5 * 128)  # 10240

## Q5-Q6: Multiple-Choice Model Outputs

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

ids, mask = tokenize_mcq_row(train.iloc[0])
with torch.no_grad():
    out = model(input_ids=ids.unsqueeze(0), attention_mask=mask.unsqueeze(0))
print('Logits shape:', out.logits.shape)  # [1, 5]
print('Number of logits:', out.logits.shape[1])  # 5

# With labels
label_tensor = torch.tensor([label2idx[train.iloc[0]['answer']]])
out_loss = model(input_ids=ids.unsqueeze(0), attention_mask=mask.unsqueeze(0), labels=label_tensor)
print('Loss dimensions:', out_loss.loss.dim())  # 0 (scalar)